# Detecting noisy monitors

This notebook shows how to use the WhyLabs Monitor Diagnoser to customize the diagnosis of a noisy monitor. It interacts with the diagnoser to get information on noisy and failing monitors, and to make selections about which monitor, segment and columns to diagnose.

## Install requirements

In [1]:
#%pip install whylabs-toolkit[diagnoser]


## Setup whylabs API connection

First, set up the information to connect to WhyLabs. Update the org_id, dataset_id and api_key in the following before running it.


In [2]:
import getpass
from whylabs_toolkit.monitor.diagnoser.helpers.utils import env_setup

org_id = 'org-0'
dataset_id = 'model-0'
api_key = getpass.getpass()
api_endpoint = 'https://songbird.development.whylabsdev.com'

env_setup(
    org_id=org_id,
    dataset_id=dataset_id,
    api_key=api_key,
    whylabs_endpoint=api_endpoint
)

Then initialize the Monitor Diagnoser with the org_id and dataset_id.

In [3]:
from whylabs_toolkit.monitor.diagnoser.monitor_diagnoser import MonitorDiagnoser
diagnoser = MonitorDiagnoser(org_id, dataset_id)

# Running a customized diagnosis
## Get the recommended diagnostic interval

Get the dataset start/end time, granularity, and a recommended diagnostic interval for the dataset. The diagnoser will use this interval unless you override it by setting the `diagnostic_interval` property.

In [4]:
lineage, granularity, interval = diagnoser.choose_dataset_batches()
lineage, granularity, interval

(TimeRange(start=datetime.datetime(2020, 10, 8, 0, 0, tzinfo=datetime.timezone.utc), end=datetime.datetime(2024, 4, 25, 21, 0, tzinfo=datetime.timezone.utc)),
 <Granularity.daily: 'daily'>,
 '2024-03-26T00:00:00.000Z/2024-04-25T00:00:00.000Z')

## Get information on noisy and failing monitors

Get information on how many anomalies are detected by each monitor in the dataset. The results are ordered so that the monitors with the most anomalies per column are first (i.e. monitors which are firing on the many batches for certain columns). Beyond that, results with a higher average number of anomalies per column are considered noisier.

In [5]:
import pandas as pd
noisy_monitors = diagnoser.detect_noisy_monitors()
noisy_monitors_df = pd.DataFrame.from_records([m.dict() for m in noisy_monitors])
noisy_monitors_df

,monitor_id,analyzer_id,metric,column_count,segment_count,anomaly_count,max_anomaly_per_column,min_anomaly_per_column,avg_anomaly_per_column,action_count,action_targets
0,kind-cyan-kangaroo-1253,kind-cyan-kangaroo-1253-analyzer,histogram,1,1,30,30,30,30,0,[]
1,cooperative-maroon-parrot-8886,discrete-drift-jensenshannon-analyzer,frequent_items,1,1,30,30,30,30,0,[]
2,famous-salmon-cobra-8902,famous-salmon-cobra-8902-analyzer,min,1,1,30,30,30,30,0,[]
3,proud-seagreen-carabeef-65,proud-seagreen-carabeef-65-analyzer,histogram,1,1,30,30,30,30,0,[]
4,None,cooperative-maroon-parrot-8886-analyzer,frequent_items,1,1,30,30,30,30,0,[]
...,...,...,...,...,...,...,...,...,...,...,...
94,glamorous-orchid-turtle-6425,glamorous-orchid-turtle-6425-analyzer,histogram,1,1,2,2,2,2,0,[]
95,breakable-limegreen-shrew-7623,breakable-limegreen-shrew-7623-analyzer,histogram,1,1,2,2,2,2,0,[]
96,hilarious-powderblue-chamois-8115,hilarious-powderblue-chamois-8115-analyzer,histogram,1,1,2,2,2,2,0,[]
97,horrible-magenta-sandpiper-8117,horrible-magenta-sandpiper-8117-analyzer,frequent_items,1,1,2,2,2,2,0,[]


Once you have run `detect_noisy_monitors`, you can retrieve the result at any time via the `noisy_monitors` property. You can also retrieve
 information about monitors with analysis failures using `failed_monitors`. 

In [6]:
failed_monitors_df = pd.DataFrame.from_records([n.dict() for n in diagnoser.failed_monitors])
failed_monitors_df

,monitor_id,analyzer_id,metric,failed_count,max_failed_per_column,min_failed_per_column,avg_failed_per_column,action_count,action_targets
0,good-cornsilk-bear-9359,good-cornsilk-bear-9359-analyzer,count_null,2310,30,30,30,0,[]
1,energetic-black-cobra-7838,energetic-black-cobra-7838-analyzer,unique_est,60,30,30,30,1,[email]
2,elated-gray-baboon-4620,elated-gray-baboon-4620-analyzer,count_null_ratio,68,30,8,22,1,[email]
3,missing-values-ratio-monitor-v9uywi,missing-values-ratio-analyzer-v9uywi,count_null_ratio,2607,25,7,24,1,[email]
4,None,expensive-tomato-moose-6522-analyzer,median,1794,23,23,23,0,[]
5,expensive-tomato-moose-6522,csw-analyzer-2,median,562,23,7,7,0,[]
6,curious-lemonchiffon-rabbit-7000,curious-lemonchiffon-rabbit-7000-analyzer,frequent_items,7,7,7,7,1,[test-sort]
7,clear-azure-starling-8883,clear-azure-starling-8883-analyzer,frequent_items,7,7,7,7,1,[test-sort]
8,light-mintcream-rhinoceros-3655,light-mintcream-rhinoceros-3655-analyzer,frequent_items,31,7,1,5,0,[]
9,handsome-lemonchiffon-eel-4222,handsome-lemonchiffon-eel-4222-analyzer,frequent_items,9,7,2,4,0,[]


From this information, the diagnoser chooses the most noisy monitor that has notification actions to diagnose. This choice can be overridden by setting the `monitor_id_to_diagnose` property of the diagnoser to the desired monitor id. 

In [7]:
diagnoser.monitor_id_to_diagnose

'kind-cyan-kangaroo-1253'

We can get the monitor object from the diagnoser, to see its display name and any other useful information.

In [8]:
diagnoser.monitor_to_diagnose

Monitor(metadata=Metadata(version=1, schemaVersion=1, updatedTimestamp=1703279098033, author='user_1759fb08_1a01_4852_9ed4_91c6fceede45', description=None), id='kind-cyan-kangaroo-1253', displayName='kind-cyan-kangaroo-1253', tags=None, analyzerIds=['kind-cyan-kangaroo-1253-analyzer'], schedule=ImmediateSchedule(type='immediate'), disabled=None, severity=3, mode=DigestMode(type='DIGEST', filter=None, creationTimeOffset=None, datasetTimestampOffset='P7D', groupBy=None), actions=[])

We can similarly see the configuration of the analyzer that is being diagnosed.


In [9]:
diagnoser.analyzer_to_diagnose

Analyzer(metadata=Metadata(version=1, schemaVersion=1, updatedTimestamp=1703279095485, author='user_1759fb08_1a01_4852_9ed4_91c6fceede45', description=None), id='kind-cyan-kangaroo-1253-analyzer', displayName=None, tags=['featureSelection:all', 'discreteness:non-discrete'], schedule=FixedCadenceSchedule(type='fixed', cadence=<Cadence.daily: 'daily'>, exclusionRanges=None), disabled=None, disableTargetRollup=None, targetMatrix=ColumnMatrix(segments=[Segment(tags=[SegmentTag(key='purpose', value='car'), SegmentTag(key='verification_status', value='Source Verified')])], type=<TargetLevel.column: 'column'>, include=[<ColumnGroups.group_continuous: 'group:continuous'>], exclude=[<ColumnGroups.group_input: 'group:input'>], profileId=None), dataReadinessDuration=None, batchCoolDownPeriod=None, backfillGracePeriodDuration=None, config=DriftConfig(schemaVersion=None, params=None, metric=<ComplexMetrics.histogram: 'histogram'>, type=<AlgorithmType.drift: 'drift'>, algorithm='hellinger', threshol

## Get information on noisy and failing segments in the analyzer

Now we use the diagnoser to get information about noisy and failing segments in the analyzer, so we can choose a segment to diagnose. The results are sorted so the segment with the most anomalies for the selected monitor is first.

In [10]:
from whylabs_toolkit.monitor.diagnoser.helpers.utils import segment_as_readable_text

noisy_segments = diagnoser.detect_noisy_segments()
noisy_segments_df = pd.DataFrame.from_records([n.dict() for n in noisy_segments])
noisy_segments_df['segment'] = [segment_as_readable_text(n.segment.tags) for n in noisy_segments]
noisy_segments_df

,segment,total_anomalies,batch_count
0,purpose=car&verification_status=Source Verified,30,30


The diagnoser chooses the noisiest segment to diagnose. This can be changed by setting the `diagnostic_segment` property.

In [11]:
segment_as_readable_text(diagnoser.diagnostic_segment.tags)

'purpose=car&verification_status=Source Verified'

## Get information on noisy columns 

The next step is to get information on the noisy columns within the segment, so we can choose a subset of columns to diagnose. 

In [12]:
noisy_columns = diagnoser.detect_noisy_columns()
noisy_columns_df = pd.DataFrame.from_records([n.dict() for n in noisy_columns])
noisy_columns_df

,column,total_anomalies
0,pred_credit_risk (output),30


The API limits diagnosis to 100 columns at a time, so we choose the top 100 noisy columns. We could then iterate through other columns if desired.

In [13]:
columns = list(noisy_columns_df.column[:100])
columns

['pred_credit_risk (output)']

## Ask for a monitor diagnosis


In [14]:
# for now, we need to enforce this to run using local server
import os
monitor_report = diagnoser.diagnose(columns)

In [15]:
print(monitor_report.describe())

Diagnosis is for monitor "kind-cyan-kangaroo-1253" [kind-cyan-kangaroo-1253] in model-0 org-0, over interval 2024-03-26T00:00:00.000Z/2024-04-25T00:00:00.000Z.

Analyzer is drift configuration for histogram metric with TrailingWindow baseline.
Analyzer "kind-cyan-kangaroo-1253-analyzer" targets 1 columns and ran on 1 columns in the diagnosed segment.


Diagnostic segment is "purpose=car&verification_status=Source Verified".
Diagnostic interval contains 30 batches.

Diagnostic interval rollup contains 10473 rows for the diagnosed columns.

Analysis results summary:
Found non-failed results for 1 columns and 30 batches.
Found 30 anomalies in 1 columns, with up to 100.0% (30) batches having anomalies per column and 100.0% (30.0) on average.
Columns with anomalies are:
|    | 0                                 |
|---:|:----------------------------------|
|  0 | ('pred_credit_risk (output)', 30) |

No failures were detected.

No issues impacting diagnosis quality were detected
Conditions tha